In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTENC
from catboost import CatBoostClassifier
from sklearn.metrics import classification_report

In [2]:
import kagglehub
import os
path = kagglehub.dataset_download("mirichoi0218/insurance")

print("Path to dataset files:", path) 
csv_file = os.path.join(path, "insurance.csv")

df = pd.read_csv(csv_file)

/Users/katiegalaeva/miniconda3/envs/catboost_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: /Users/katiegalaeva/.cache/kagglehub/datasets/mirichoi0218/insurance/versions/1


In [3]:
df_train = pd.read_csv('train.csv')
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 786431 entries, 0 to 786430
Data columns (total 18 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   transaction_time  786431 non-null  object 
 1   merch             786431 non-null  object 
 2   cat_id            786431 non-null  object 
 3   amount            786431 non-null  float64
 4   name_1            786431 non-null  object 
 5   name_2            786431 non-null  object 
 6   gender            786431 non-null  object 
 7   street            786431 non-null  object 
 8   one_city          786431 non-null  object 
 9   us_state          786431 non-null  object 
 10  post_code         786431 non-null  int64  
 11  lat               786431 non-null  float64
 12  lon               786431 non-null  float64
 13  population_city   786431 non-null  int64  
 14  jobs              786431 non-null  object 
 15  merchant_lat      786431 non-null  float64
 16  merchant_lon      78

In [4]:
df_train['target'].value_counts()

target
0    781927
1      4504
Name: count, dtype: int64

In [5]:
df_test = pd.read_csv('test.csv')
df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 262144 entries, 0 to 262143
Data columns (total 17 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   transaction_time  262144 non-null  object 
 1   merch             262144 non-null  object 
 2   cat_id            262144 non-null  object 
 3   amount            262144 non-null  float64
 4   name_1            262144 non-null  object 
 5   name_2            262144 non-null  object 
 6   gender            262144 non-null  object 
 7   street            262144 non-null  object 
 8   one_city          262144 non-null  object 
 9   us_state          262144 non-null  object 
 10  post_code         262144 non-null  int64  
 11  lat               262144 non-null  float64
 12  lon               262144 non-null  float64
 13  population_city   262144 non-null  int64  
 14  jobs              262144 non-null  object 
 15  merchant_lat      262144 non-null  float64
 16  merchant_lon      26

In [6]:
df_train

,transaction_time,merch,cat_id,amount,name_1,name_2,gender,street,one_city,us_state,post_code,lat,lon,population_city,jobs,merchant_lat,merchant_lon,target
0,2019-12-27 15:21,fraud_Cormier LLC,health_fitness,148.04,Daniel,Martinez,M,8510 Acevedo Burgs,Kent,OR,97033,45.0838,-120.6649,60,Museum education officer,45.042827,-120.709327,0
1,2019-04-17 23:09,"fraud_Brown, Homenick and Lesch",health_fitness,39.40,Grace,Williams,F,28812 Charles Mill Apt. 628,Plantersville,AL,36758,32.6176,-86.9475,1412,Drilling engineer,31.872266,-87.828247,0
2,2019-09-23 15:02,fraud_Ruecker-Mayert,kids_pets,52.96,Kyle,Park,M,7507 Larry Passage Suite 859,Mount Perry,OH,43760,39.8788,-82.1880,1831,Barrister's clerk,40.010874,-81.841249,0
3,2019-05-13 16:00,"fraud_Mante, Luettgen and Hackett",health_fitness,7.66,Monique,Martin,F,68276 Matthew Springs,Ratcliff,TX,75858,31.3833,-95.0619,43,"Engineer, production",30.888406,-95.141609,0
4,2019-08-18 07:27,fraud_Luettgen PLC,gas_transport,51.59,Christine,Johnson,F,8011 Chapman Tunnel Apt. 568,Blairsden-Graeagle,CA,96103,39.8127,-120.6405,1725,Chartered legal executive (England and Wales),39.376017,-121.311691,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
786426,2019-04-10 12:35,"fraud_O'Connell, Botsford and Hand",home,76.56,Ryan,Foster,M,03921 Cole Mission Suite 882,Hampton,FL,32044,29.8575,-82.1483,2060,Oncologist,29.235257,-82.407844,0
786427,2019-12-15 09:34,"fraud_Adams, Kovacek and Kuhlman",grocery_net,68.58,Jim,Johnson,M,868 Brady Mill Apt. 837,Gretna,LA,70056,29.8872,-90.0331,55581,Biomedical scientist,29.015274,-90.564712,0
786428,2019-10-12 10:22,"fraud_Lind, Huel and McClure",gas_transport,66.66,Christopher,Horn,M,956 Sanchez Highway,Mallie,KY,41836,37.2692,-82.9161,798,Facilities manager,37.515508,-82.443788,0
786429,2019-10-18 09:01,fraud_Rempel PLC,grocery_net,38.06,Samuel,Sandoval,M,0005 Morrison Land,Mounds,OK,74047,35.8896,-96.0887,7163,Fitness centre manager,35.203864,-96.999902,0


In [7]:
df_train.describe()

,amount,post_code,lat,lon,population_city,merchant_lat,merchant_lon,target
count,786431.000000,786431.000000,786431.000000,786431.000000,7.864310e+05,786431.000000,786431.000000,786431.000000
mean,70.241296,48802.521336,38.527972,-90.224069,8.928853e+04,38.527301,-90.224508,0.005727
std,161.091489,26896.564152,5.078756,13.754760,3.028600e+05,5.113222,13.766977,0.075461
min,1.000000,1257.000000,20.027100,-165.672300,2.300000e+01,19.027804,-166.670132,0.000000
25%,9.650000,26237.000000,34.620500,-96.798000,7.430000e+02,34.727480,-96.901593,0.000000
50%,47.410000,48174.000000,39.346500,-87.476900,2.456000e+03,39.357665,-87.436919,0.000000
75%,83.000000,72042.000000,41.894800,-80.158000,2.047800e+04,41.950609,-80.233429,0.000000
max,27390.120000,99783.000000,66.693300,-67.950300,2.906700e+06,67.441518,-66.955996,1.000000


In [8]:
df_train['transaction_time'] = pd.to_datetime(df_train['transaction_time'])

df_train['year'] = df_train['transaction_time'].dt.year
df_train['month'] = df_train['transaction_time'].dt.month
df_train['day'] = df_train['transaction_time'].dt.day
df_train['hour'] = df_train['transaction_time'].dt.hour
df_train['minute'] = df_train['transaction_time'].dt.minute
df_train['day_of_week'] = df_train['transaction_time'].dt.dayofweek

In [9]:
df_test['transaction_time'] = pd.to_datetime(df_test['transaction_time'])

df_test['year'] = df_test['transaction_time'].dt.year
df_test['month'] = df_test['transaction_time'].dt.month
df_test['day'] = df_test['transaction_time'].dt.day
df_test['hour'] = df_test['transaction_time'].dt.hour
df_test['minute'] = df_test['transaction_time'].dt.minute
df_test['day_of_week'] = df_test['transaction_time'].dt.dayofweek

In [10]:
numeric_features = [
    'amount', 'ts_transaction_time', 'lat', 'lon', 'population_city', 'merchant_lat', 'merchant_lon'
]

cat_columns = [
    'merch', 'cat_id', 'name_1', 'name_2', 'gender', 'street', 'one_city', 'us_state', 'post_code', 'jobs', 'year',	'month',	'day',	'hour',	'minute'	,'day_of_week'
]

In [11]:
import numpy as np
from math import atan2, cos, radians, sin, sqrt


def haversine_distance(lat1: float, lon1: float, lat2: float, lon2: float, n_digits: int = 0) -> float:
    """
        Функция для расчёта расстояния от точки А до Б по прямой

        :param lat1: Широта точки А
        :param lon1: Долгота точки А
        :param lat2: Широта точки Б
        :param lon2: Долгота точки Б
        :param n_digits: Округляем полученный ответ до n знака после запятой
        :return: Дистанция по прямой с точностью до n_digits
    """

    lat1, lon1, lat2, lon2 = round(lat1, 6), round(lon1, 6), round(lat2, 6), round(lon2, 6)
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)

    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2) ** 2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2) ** 2

    return round(2 * 6372800 * np.arctan2(np.sqrt(a), np.sqrt(1 - a)), n_digits)  # метры.сантиметры


def bearing_degree(lat1: float, lon1: float, lat2: float, lon2: float, n_digits: int = 0) -> float:
    """
        Функция для расчёта угла между прямой [((lat1, lon1), (lat2, lon2)), (нулевой мередиан)]

        :param lat1: Широта точки А
        :param lon1: Долгота точки А
        :param lat2: Широта точки Б
        :param lon2: Долгота точки Б
        :param n_digits: Округляем полученный ответ до n знака после запятой
        :return: Значение угла с точностью до n_digits
    """

    lat1, lon1 = np.radians(round(lat1, 6)), np.radians(round(lon1, 6))
    lat2, lon2 = np.radians(round(lat2, 6)), np.radians(round(lon2, 6))

    dlon = (lon2 - lon1)
    numerator = np.sin(dlon) * np.cos(lat2)
    denominator = np.cos(lat1) * np.sin(lat2) - (np.sin(lat1) * np.cos(lat2) * np.cos(dlon))

    theta = np.arctan2(numerator, denominator)
    theta_deg = (np.degrees(theta) + 360) % 360

    return round(theta_deg, n_digits)

In [12]:
df_train['bearing_degree_1'] = bearing_degree(df_train['lat'], df_train['lon'], df_train['merchant_lat'], df_train['merchant_lon'], ).values
df_test['bearing_degree_1'] = bearing_degree(df_test['lat'], df_test['lon'], df_test['merchant_lat'], df_test['merchant_lon'], ).values

In [13]:
df_train['bearing_degree_2'] = bearing_degree(df_train['lat'], df_train['lon'], 0, 0, ).values
df_test['bearing_degree_2'] = bearing_degree(df_test['lat'], df_test['lon'], 0, 0, ).values

df_train['bearing_degree_3'] = bearing_degree(0, 0, df_train['merchant_lat'], df_train['merchant_lon'], ).values
df_test['bearing_degree_3'] = bearing_degree(0, 0, df_test['merchant_lat'], df_test['merchant_lon'], ).values

In [14]:
df_train['hav_dist_1'] = haversine_distance(df_train['lat'], df_train['lon'], df_train['merchant_lat'], df_train['merchant_lon'], ).values
df_test['hav_dist_1'] = haversine_distance(df_test['lat'], df_test['lon'], df_test['merchant_lat'], df_test['merchant_lon'], ).values
df_train['hav_dist_2'] = haversine_distance(df_train['lat'], df_train['lon'], 0, 0, ).values
df_test['hav_dist_2'] = haversine_distance(df_test['lat'], df_test['lon'], 0, 0, ).values

df_train['hav_dist_3'] = haversine_distance(0, 0, df_train['merchant_lat'], df_train['merchant_lon'], ).values
df_test['hav_dist_3'] = haversine_distance(0, 0, df_test['merchant_lat'], df_test['merchant_lon'], ).values

In [17]:
model_features = [
  'amount',
  'bearing_degree_1',
  'bearing_degree_2',
  'bearing_degree_3',
  'cat_id',
  'gender',
  'hav_dist_1',
  'hav_dist_2',
  'hav_dist_3',
  'jobs',
  'lat',
  'lon',
  'merch',
  'merchant_lat',
  'merchant_lon',
  'name_1',
  'name_2',
  'one_city',
  'population_city',
  'post_code',
  'street',
  'us_state',
  'year',	
  'month',	
  'day',	
  'hour',	
  'minute',
  'day_of_week'
]

In [39]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Разделение на признаки и таргет
X = df_train.drop('target', axis=1)
y = df_train['target']

# Разделение на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(X[model_features], y, test_size=0.2, random_state=42)

# Применение target encoding для категориальных признаков
# Для каждого категориального признака вычисляем среднее значение таргета в обучающей выборке
for col in cat_columns:
    # Создаём mapping из категорий в среднее значение таргета
    mapping = X_train.join(y_train).groupby(col)['target'].mean()
    # Создаём новые признаки с суффиксом '_te'
    X_train[col + '_te'] = X_train[col].map(mapping)
    X_test[col + '_te'] = X_test[col].map(mapping)
    

# Если исходные категориальные признаки больше не нужны, их можно удалить:
X_train.drop(columns=cat_columns, inplace=True)
X_test.drop(columns=cat_columns, inplace=True)
df_test.drop(columns=cat_columns, inplace=True)

# Теперь нормализуем все числовые признаки
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
df_test_scaled = scaler.transform(df_test)

# Результат: X_train_scaled и X_test_scaled готовы для обучения модели
print("Форма обучающей выборки:", X_train_scaled.shape)
print("Форма тестовой выборки:", X_test_scaled.shape)


KeyError: "['merch', 'cat_id', 'name_1', 'name_2', 'gender', 'street', 'one_city', 'us_state', 'post_code', 'jobs', 'year', 'month', 'day', 'hour', 'minute', 'day_of_week'] not found in axis"

In [42]:
scaler.fit(df_test)
df_test_scaled = scaler.transform(df_test)

/Users/katiegalaeva/miniconda3/envs/catboost_env/lib/python3.12/site-packages/sklearn/utils/extmath.py:1101: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/Users/katiegalaeva/miniconda3/envs/catboost_env/lib/python3.12/site-packages/sklearn/utils/extmath.py:1106: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
/Users/katiegalaeva/miniconda3/envs/catboost_env/lib/python3.12/site-packages/sklearn/utils/extmath.py:1126: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count


In [29]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# Предполагается, что X_train_scaled, y_train, X_test_scaled, y_test уже созданы (массивы NumPy)
# Преобразуем данные в тензоры PyTorch
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values if hasattr(y_train, 'values') else y_train, dtype=torch.float32).unsqueeze(1)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values if hasattr(y_test, 'values') else y_test, dtype=torch.float32).unsqueeze(1)

# Создаём TensorDataset и DataLoader для обучения и тестирования
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

# Определяем базовую модель нейросети
class BinaryClassificationModel(nn.Module):
    def __init__(self, input_dim):
        super(BinaryClassificationModel, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 1),
            nn.Sigmoid()  # для бинарной классификации, возвращает вероятность
        )
        
    def forward(self, x):
        return self.net(x)

# Инициализация модели, функции потерь и оптимизатора
input_dim = 28  # число признаков
model = BinaryClassificationModel(input_dim)

criterion = nn.BCELoss()  # бинарная кросс-энтропия
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Если доступен GPU, переводим модель на устройство
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Цикл обучения
num_epochs = 1
for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0.0
    correct = 0
    total = 0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item() * batch_X.size(0)
        
        # Расчёт точности: если вероятность > 0.5, то класс 1
        preds = (outputs > 0.5).float()
        correct += (preds == batch_y).sum().item()
        total += batch_y.size(0)
    
    epoch_loss /= total
    accuracy = correct / total
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}, Accuracy: {accuracy:.4f}")
    
    # Оценка модели на тестовом наборе
    model.eval()
    test_loss = 0.0
    correct_test = 0
    total_test = 0
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            test_loss += loss.item() * batch_X.size(0)
            preds = (outputs > 0.5).float()
            correct_test += (preds == batch_y).sum().item()
            total_test += batch_y.size(0)
    test_loss /= total_test
    test_accuracy = correct_test / total_test
    print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.4f}")


Epoch 1/1, Loss: 0.0256, Accuracy: 0.9954
Test Loss: 0.0181, Test Accuracy: 0.9959


In [50]:
df_test_tensor = torch.tensor(df_test_scaled, dtype=torch.float32)


In [51]:
y_pred = model(df_test_tensor)

In [58]:
numpy_array = y_pred.detach().numpy()

In [56]:
submission = pd.DataFrame({'prediction': numpy_array})

submission = submission.reset_index()

submission.rename(columns={'index': 'index', 'prediction': 'prediction'}, inplace=True)

submission.to_csv("submission_2.csv", index=False)

ValueError: Per-column arrays must each be 1-dimensional

In [73]:
submission['prediction'].value_counts()

prediction
0    261730
1       414
Name: count, dtype: int64